# 03 — PCA exploration and preprocessing selection

This notebook compares PCA representations for the NIR UCO project.

Main objectives:

1. Load the validated NIR UCO database.
2. Select pure almond and pure peanut objects.
3. Compare matrix representations:
   - `object_mean`
   - `object_median`
   - `balanced_pixels_random`
   - `balanced_pixels_center`
   - optional: `all_pixels`
4. Compare spectral preprocessing methods:
   - raw
   - absorbance
   - SNV / MSC
   - Savitzky-Golay smoothing
   - Savitzky-Golay derivative
   - combined chains
5. Rank preprocessing × matrix combinations using PCA diagnostics.
6. Select candidate preprocessings for downstream SIMCA and MCR analysis.

This notebook is exploratory. It does not select the final SIMCA model.

In [2]:
from __future__ import annotations

import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 180)
pd.set_option("display.max_rows", 250)

# ---------------------------------------------------------------------
# Project root detection
# ---------------------------------------------------------------------
CURRENT_DIR = Path.cwd().resolve()

if (CURRENT_DIR / "src").exists():
    PROJECT_ROOT = CURRENT_DIR
elif (CURRENT_DIR.parent / "src").exists():
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    raise RuntimeError(
        "Could not find project root. Launch the notebook from the project "
        "root or from the notebooks/ folder."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("PROJECT_ROOT:", PROJECT_ROOT)

PROJECT_ROOT: C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts


In [3]:
from src.io.database_h5 import load_nir_uco_h5

from src.utils import (
    save_parquet,
)

from src.matrices.matrix_registry import build_matrix

from src.spectra.preprocessing_configs import normalize_preprocessing_configs
from src.spectra.band_selection import (
    select_wavelength_range_from_database,
    wavelength_selection_summary,
)

from src.workflows.pca import (
    compare_pca_representations,
    add_pca_selection_score,
)

from src.visualization.plot_pca import (
    plot_explained_variance,
    plot_loadings,
    plot_pca_diagnostic,
    plot_pca_metric_heatmap,
    plot_pca_metric_tradeoff,
    plot_pca_metric_ranking,
)

from src.visualization.plot_scores import (
    plot_scores,
    build_scores_dataframe,
    sample_scores_dataframe,
    plot_scores_density,
    summarize_scores_by_object,
    plot_object_score_summary,
)
from src.visualization.plot_spectra import plot_spectra

%load_ext autoreload
%autoreload 2

In [4]:
# ---------------------------------------------------------------------
# Input database
# ---------------------------------------------------------------------
DB_H5_PATH = (
    PROJECT_ROOT
    / "HSI Data"
    / "processed"
    / "nir_uco_database.h5"
)

# ---------------------------------------------------------------------
# Spectral configuration
# ---------------------------------------------------------------------
# Current workflow:
#   use all non-noisy bands stored in the H5 database.
#
# Later, set USE_WAVELENGTH_WINDOW=True to test a spectral window.
USE_WAVELENGTH_WINDOW = False

WAVELENGTH_MODE = "non_noisy_all"

WINDOW_MIN_NM = 1225.0
WINDOW_MAX_NM = 1675.0

if USE_WAVELENGTH_WINDOW:
    RESULTS_TAG = f"{int(WINDOW_MIN_NM)}_{int(WINDOW_MAX_NM)}"
else:
    RESULTS_TAG = "non_noisy_all"

# ---------------------------------------------------------------------
# Outputs
# ---------------------------------------------------------------------
RESULTS_DIR = PROJECT_ROOT / "results" / f"03_pca_{RESULTS_TAG}"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

PCA_SUMMARY_PATH = RESULTS_DIR / "pca_summary.parquet"
PCA_SELECTED_PREPROCESSINGS_PATH = RESULTS_DIR / "pca_selected_preprocessings.parquet"

# ---------------------------------------------------------------------
# PCA data subset
# ---------------------------------------------------------------------
TARGET_CLASS = "peanut"
REFERENCE_CLASSES = ("almond", TARGET_CLASS)

# Use pure objects only for PCA exploration.
# This avoids mixing true pure samples with position-reference images.
PCA_SAMPLE_KIND = "pure"

# Use all pure batches for exploration.
# Later SIMCA notebooks will use a stricter train/validation/test protocol.
PCA_ALLOWED_BATCHES = [1, 2, 3, 4]

# ---------------------------------------------------------------------
# PCA comparison settings
# ---------------------------------------------------------------------
N_COMPONENTS = 20

M_BALANCED_PIXELS = 40
REPLACE_BALANCED_PIXELS = False
RANDOM_STATE = 42

BALANCED_PIXEL_STRATEGIES = ["random", "center"]

# Optional because all_pixels can become large.
RUN_ALL_PIXELS = False

# ---------------------------------------------------------------------
# Preprocessing settings
# ---------------------------------------------------------------------
SG_WINDOW_LENGTH = 11
SG_POLYORDER = 2

PREPROCESSING_METHODS = {
    "raw": ("raw",),
    "absorbance": ("absorbance",),
    "snv": ("snv",),
    "msc": ("msc",),
    "sg_smooth": ("sg_smooth",),
    "sg_d1": ("sg_d1",),
    "sg_d2": ("sg_d2",),
    "absorbance_snv": ("absorbance", "snv"),
    "absorbance_msc": ("absorbance", "msc"),
    "absorbance_sg_smooth": ("absorbance", "sg_smooth"),
    "absorbance_sg_d1": ("absorbance", "sg_d1"),
    "absorbance_sg_d2": ("absorbance", "sg_d2"),
    "snv_sg_smooth": ("snv", "sg_smooth"),
    "snv_sg_d1": ("snv", "sg_d1"),
    "snv_sg_d2": ("snv", "sg_d2"),
    "absorbance_snv_sg_smooth": ("absorbance", "snv", "sg_smooth"),
    "absorbance_snv_sg_d1": ("absorbance", "snv", "sg_d1"),
    "absorbance_snv_sg_d2": ("absorbance", "snv", "sg_d2"),
}

# ---------------------------------------------------------------------
# Plotting settings
# ---------------------------------------------------------------------
N_TOP_TO_DISPLAY = 20
N_TOP_TO_PLOT = 5
N_SHORTLIST_PER_MATRIX_VARIANT = 5
N_GLOBAL_SHORTLIST = 10

RUN_SPECTRA_CHECK_PLOTS = True
RUN_DETAILED_PCA_PLOTS = True
MAX_SPECTRA_TO_PLOT = 80

print("DB_H5_PATH:", DB_H5_PATH)
print("RESULTS_DIR:", RESULTS_DIR)
print("WAVELENGTH_MODE:", WAVELENGTH_MODE)
print("USE_WAVELENGTH_WINDOW:", USE_WAVELENGTH_WINDOW)
print("RESULTS_TAG:", RESULTS_TAG)
print("RUN_ALL_PIXELS:", RUN_ALL_PIXELS)

DB_H5_PATH: C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\HSI Data\processed\nir_uco_database.h5
RESULTS_DIR: C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\03_pca_non_noisy_all
WAVELENGTH_MODE: non_noisy_all
USE_WAVELENGTH_WINDOW: False
RESULTS_TAG: non_noisy_all
RUN_ALL_PIXELS: False


In [5]:
object_db, image_db = load_nir_uco_h5(
    DB_H5_PATH,
    reconstruct_heavy_object_arrays=True,
)

# ---------------------------------------------------------------------
# Wavelength handling
# ---------------------------------------------------------------------
if USE_WAVELENGTH_WINDOW:
    object_db, image_db, wavelengths, wavelength_info = select_wavelength_range_from_database(
        object_db=object_db,
        image_db=image_db,
        min_nm=WINDOW_MIN_NM,
        max_nm=WINDOW_MAX_NM,
    )

    wavelength_selection_df = wavelength_selection_summary(wavelength_info)

else:
    first_object = next(iter(object_db.values()))
    wavelengths = first_object.get("wavelengths")
    wavelengths = np.asarray(wavelengths) if wavelengths is not None else None
    wavelength_selection_df = pd.DataFrame()

if wavelengths is not None and len(wavelengths) > 0:
    wavelength_config_df = pd.DataFrame([{
        "wavelength_mode": WAVELENGTH_MODE,
        "use_wavelength_window": bool(USE_WAVELENGTH_WINDOW),
        "results_tag": RESULTS_TAG,
        "window_min_nm": WINDOW_MIN_NM if USE_WAVELENGTH_WINDOW else np.nan,
        "window_max_nm": WINDOW_MAX_NM if USE_WAVELENGTH_WINDOW else np.nan,
        "n_bands": int(len(wavelengths)),
        "min_wavelength_nm": float(np.min(wavelengths)),
        "max_wavelength_nm": float(np.max(wavelengths)),
    }])
else:
    wavelength_config_df = pd.DataFrame([{
        "wavelength_mode": WAVELENGTH_MODE,
        "use_wavelength_window": bool(USE_WAVELENGTH_WINDOW),
        "results_tag": RESULTS_TAG,
        "window_min_nm": WINDOW_MIN_NM if USE_WAVELENGTH_WINDOW else np.nan,
        "window_max_nm": WINDOW_MAX_NM if USE_WAVELENGTH_WINDOW else np.nan,
        "n_bands": np.nan,
        "min_wavelength_nm": np.nan,
        "max_wavelength_nm": np.nan,
    }])

print("Wavelength configuration:")
display(wavelength_config_df)

if USE_WAVELENGTH_WINDOW:
    print("Detailed wavelength selection:")
    display(wavelength_selection_df)

Wavelength configuration:


,wavelength_mode,use_wavelength_window,results_tag,window_min_nm,window_max_nm,n_bands,min_wavelength_nm,max_wavelength_nm
0,non_noisy_all,False,non_noisy_all,NaN,NaN,63,960.735294,1702.0


In [6]:
object_rows = []

for object_id, obj in object_db.items():
    object_rows.append({
        "object_id": object_id,
        "source_clean_key": obj.get("source_clean_key"),
        "source_image": obj.get("source_image"),
        "sample_kind": obj.get("sample_kind"),
        "object_nut_type": obj.get("object_nut_type"),
        "image_nut_type": obj.get("image_nut_type"),
        "batch": obj.get("batch"),
        "split": obj.get("split"),
        "area_pixels": obj.get("area_pixels"),
        "n_pixels": obj.get("n_pixels"),
        "n_bands": obj.get("n_bands"),
        "is_pure": obj.get("is_pure"),
        "is_mixture": obj.get("is_mixture"),
        "is_position_reference": obj.get("is_position_reference"),
    })

object_meta_df = pd.DataFrame(object_rows)

display(object_meta_df.head())

display(
    object_meta_df
    .groupby(["sample_kind", "object_nut_type", "batch"], dropna=False)
    .size()
    .reset_index(name="n_objects")
    .sort_values(["sample_kind", "object_nut_type", "batch"], na_position="last")
)

,object_id,source_clean_key,source_image,sample_kind,object_nut_type,image_nut_type,batch,split,area_pixels,n_pixels,n_bands,is_pure,is_mixture,is_position_reference
0,alm1pea1_obj001,alm1pea1,alm1pea1_sb,mixture,unknown,mixture,NaN,projection,84,84,63,False,True,False
1,alm1pea1_obj002,alm1pea1,alm1pea1_sb,mixture,unknown,mixture,NaN,projection,73,73,63,False,True,False
2,alm1pea1_obj003,alm1pea1,alm1pea1_sb,mixture,unknown,mixture,NaN,projection,91,91,63,False,True,False
3,alm1pea1_obj004,alm1pea1,alm1pea1_sb,mixture,unknown,mixture,NaN,projection,73,73,63,False,True,False
4,alm1pea1_obj005,alm1pea1,alm1pea1_sb,mixture,unknown,mixture,NaN,projection,126,126,63,False,True,False


,sample_kind,object_nut_type,batch,n_objects
0,mixture,unknown,NaN,722
1,position_reference,peanut,1.0,47
2,position_reference,peanut,2.0,47
3,position_reference,peanut,3.0,47
4,position_reference,peanut,4.0,5
5,pure,almond,1.0,52
6,pure,almond,2.0,59
7,pure,almond,3.0,55
8,pure,almond,4.0,48
9,pure,peanut,1.0,46


In [7]:
def subset_object_db_for_pca(
    object_db,
    sample_kind="pure",
    REFERENCE_CLASSES=("almond", "peanut"),
    allowed_batches=None,
):
    """Return a filtered object_db for PCA exploration."""
    out = {}

    REFERENCE_CLASSES = set(map(str, REFERENCE_CLASSES))

    if allowed_batches is not None:
        allowed_batches = set(allowed_batches)

    for object_id, obj in object_db.items():
        if str(obj.get("sample_kind")) != str(sample_kind):
            continue

        if str(obj.get("object_nut_type")) not in REFERENCE_CLASSES:
            continue

        if allowed_batches is not None:
            batch = obj.get("batch")
            if batch not in allowed_batches:
                continue

        out[object_id] = obj

    return out


object_db_pca = subset_object_db_for_pca(
    object_db=object_db,
    sample_kind=PCA_SAMPLE_KIND,
    REFERENCE_CLASSES=REFERENCE_CLASSES,
    allowed_batches=PCA_ALLOWED_BATCHES,
)

print(f"Objects selected for PCA: {len(object_db_pca)}")

pca_object_meta_df = object_meta_df[
    object_meta_df["object_id"].isin(object_db_pca.keys())
].copy()

display(
    pca_object_meta_df
    .groupby(["sample_kind", "object_nut_type", "batch"], dropna=False)
    .size()
    .reset_index(name="n_objects")
    .sort_values(["object_nut_type", "batch"], na_position="last")
)

if len(object_db_pca) == 0:
    raise RuntimeError("No object selected for PCA. Check filtering parameters.")

Objects selected for PCA: 394


,sample_kind,object_nut_type,batch,n_objects
0,pure,almond,1.0,52
1,pure,almond,2.0,59
2,pure,almond,3.0,55
3,pure,almond,4.0,48
4,pure,peanut,1.0,46
5,pure,peanut,2.0,52
6,pure,peanut,3.0,53
7,pure,peanut,4.0,29


In [8]:
first_object = next(iter(object_db_pca.values()))
wavelengths = first_object.get("wavelengths")

if wavelengths is not None:
    wavelengths = np.asarray(wavelengths)
    if wavelengths.size == 0:
        wavelengths = None

if wavelengths is None:
    print("No wavelength axis found. PCA plots will use band indices.")
else:
    print("Wavelength axis found.")
    print("n_wavelengths:", len(wavelengths))
    print("first:", wavelengths[:5])
    print("last:", wavelengths[-5:])

Wavelength axis found.
n_wavelengths: 63
first: [ 960.73529412  972.69117647  984.64705882  996.60294118 1008.55882353]
last: [1654.17647059 1666.13235294 1678.08823529 1690.04411765 1702.        ]


In [9]:
X_mean_raw, y_mean_raw, meta_mean_raw = build_matrix(
    object_db=object_db_pca,
    matrix_method="object_mean",
    filters={
        "object_nut_type": list(REFERENCE_CLASSES),
    },
)

meta_mean_raw_df = pd.DataFrame(meta_mean_raw)

print("X_mean_raw:", X_mean_raw.shape)
print("y_mean_raw:", y_mean_raw.shape)
display(meta_mean_raw_df.head())

# Optional sampling for detailed plot.
if RUN_SPECTRA_CHECK_PLOTS:
    # Optional sampling for detailed plot.
    rng = np.random.default_rng(RANDOM_STATE)

    if X_mean_raw.shape[0] > MAX_SPECTRA_TO_PLOT:
        sampled_indices = []

        temp_df = meta_mean_raw_df.copy()
        temp_df["_row_index"] = np.arange(len(temp_df))
        temp_df["label"] = y_mean_raw

        for label, sub in temp_df.groupby("label", dropna=False):
            n = min(MAX_SPECTRA_TO_PLOT // len(REFERENCE_CLASSES), len(sub))
            sampled = sub.sample(n=n, random_state=RANDOM_STATE)
            sampled_indices.extend(sampled["_row_index"].tolist())

        sampled_indices = sorted(sampled_indices)
    else:
        sampled_indices = np.arange(X_mean_raw.shape[0])

    plot_spectra(
        X_mean_raw[sampled_indices],
        wavelengths=wavelengths,
        labels=y_mean_raw[sampled_indices],
        names=meta_mean_raw_df.iloc[sampled_indices]["object_id"].to_numpy()
            if "object_id" in meta_mean_raw_df.columns
            else None,
        reducer="none",
        title="Raw object mean spectra — sampled pure objects",
        y_title="Reflectance",
        show=True,
    )

    plot_spectra(
        X_mean_raw,
        wavelengths=wavelengths,
        labels=y_mean_raw,
        reducer="mean_std",
        title="Raw object mean spectra — mean ± std by class",
        y_title="Reflectance",
        show=True,
    )
else:
    print("Spectral check plots skipped.")

X_mean_raw: (394, 63)
y_mean_raw: (394,)


,object_id,label,source_image,batch,area,sample_kind
0,almond1_obj001,almond,almond1,1,54,pure
1,almond1_obj002,almond,almond1,1,95,pure
2,almond1_obj003,almond,almond1,1,52,pure
3,almond1_obj004,almond,almond1,1,98,pure
4,almond1_obj005,almond,almond1,1,84,pure


## 1. PCA comparison design

We compare several matrix representations.

To avoid mixing pure samples with position-reference images, the PCA input object database has already been filtered to pure almond and pure peanut objects only.

Because the corrected database currently stores all objects with `split="projection"` for backward compatibility with the previous script, this notebook does not filter by split.

In [10]:
pca_runs = [
    {
        "run_id": "object_matrices",
        "matrix_methods": ["object_mean", "object_median"],
        "balanced_pixel_strategy": "random",
    },
    {
        "run_id": "balanced_pixels_random",
        "matrix_methods": ["balanced_pixels"],
        "balanced_pixel_strategy": "random",
    },
]

if "center" in BALANCED_PIXEL_STRATEGIES:
    pca_runs.append({
        "run_id": "balanced_pixels_center",
        "matrix_methods": ["balanced_pixels"],
        "balanced_pixel_strategy": "center",
    })

if RUN_ALL_PIXELS:
    pca_runs.append({
        "run_id": "all_pixels",
        "matrix_methods": ["all_pixels"],
        "balanced_pixel_strategy": "random",
    })

pca_runs_df = pd.DataFrame(pca_runs)
pca_runs_df

,run_id,matrix_methods,balanced_pixel_strategy
0,object_matrices,"[object_mean, object_median]",random
1,balanced_pixels_random,[balanced_pixels],random
2,balanced_pixels_center,[balanced_pixels],center


In [11]:
valid_preprocessing_configs = normalize_preprocessing_configs(PREPROCESSING_METHODS)
valid_preprocessing_configs

{'raw': ('raw',),
 'absorbance': ('absorbance',),
 'snv': ('snv',),
 'msc': ('msc',),
 'sg_smooth': ('sg_smooth',),
 'sg_d1': ('sg_d1',),
 'sg_d2': ('sg_d2',),
 'absorbance_snv': ('absorbance', 'snv'),
 'absorbance_msc': ('absorbance', 'msc'),
 'absorbance_sg_smooth': ('absorbance', 'sg_smooth'),
 'absorbance_sg_d1': ('absorbance', 'sg_d1'),
 'absorbance_sg_d2': ('absorbance', 'sg_d2'),
 'snv_sg_smooth': ('snv', 'sg_smooth'),
 'snv_sg_d1': ('snv', 'sg_d1'),
 'snv_sg_d2': ('snv', 'sg_d2'),
 'absorbance_snv_sg_smooth': ('absorbance', 'snv', 'sg_smooth'),
 'absorbance_snv_sg_d1': ('absorbance', 'snv', 'sg_d1'),
 'absorbance_snv_sg_d2': ('absorbance', 'snv', 'sg_d2')}

In [12]:
pca_summary_parts = []
pca_results_registry = {}

for run in pca_runs:
    run_id = run["run_id"]
    matrix_methods = run["matrix_methods"]
    balanced_pixel_strategy = run["balanced_pixel_strategy"]

    print("=" * 100)
    print("PCA run:", run_id)
    print("matrix_methods:", matrix_methods)
    print("balanced_pixel_strategy:", balanced_pixel_strategy)
    print("=" * 100)

    try:
        summary_run_df, results_run = compare_pca_representations(
            object_db=object_db_pca,
            matrix_methods=matrix_methods,
            preprocessing_methods=valid_preprocessing_configs,
            allowed_splits=None,  # Important: the database was built with forced split="projection"
            allowed_labels=REFERENCE_CLASSES,
            n_components=N_COMPONENTS,
            m=M_BALANCED_PIXELS,
            wavelengths=wavelengths,
            random_state=RANDOM_STATE,
            replace=REPLACE_BALANCED_PIXELS,
            sg_window_length=SG_WINDOW_LENGTH,
            sg_polyorder=SG_POLYORDER,
            balanced_pixel_strategy=balanced_pixel_strategy,
        )

        summary_run_df["run_id"] = run_id

        # Backward compatibility if an old version of src.workflows.pca is loaded.
        if "balanced_pixel_strategy" not in summary_run_df.columns:
            summary_run_df["balanced_pixel_strategy"] = np.where(
                summary_run_df["matrix_method"].eq("balanced_pixels"),
                balanced_pixel_strategy,
                "not_applicable",
            )
        if "balanced_pixel_strategy_effective" not in summary_run_df.columns:
            summary_run_df["balanced_pixel_strategy_effective"] = balanced_pixel_strategy
        if "matrix_family" not in summary_run_df.columns:
            summary_run_df["matrix_family"] = np.where(
                summary_run_df["matrix_method"].isin(["object_mean", "object_median"]),
                "object_matrix",
                "pixel_matrix",
            )
        if "matrix_variant" not in summary_run_df.columns:
            summary_run_df["matrix_variant"] = np.where(
                summary_run_df["matrix_method"].eq("balanced_pixels"),
                "balanced_pixels_" + summary_run_df["balanced_pixel_strategy_effective"].astype(str),
                summary_run_df["matrix_method"].astype(str),
            )

        pca_summary_parts.append(summary_run_df)
        pca_results_registry[run_id] = results_run

    except Exception as exc:
        print(f"[ERROR] PCA run failed: {run_id}")
        print(repr(exc))
        raise

pca_summary_df = pd.concat(pca_summary_parts, ignore_index=True)

display(pca_summary_df.head())
print("PCA summary shape:", pca_summary_df.shape)

PCA run: object_matrices
matrix_methods: ['object_mean', 'object_median']
balanced_pixel_strategy: random

=== Matrix method: object_mean ===
X shape: (394, 63)
Labels: {'almond': 214, 'peanut': 180}
  - preprocessing: raw
  - preprocessing: absorbance
  - preprocessing: snv
  - preprocessing: msc
  - preprocessing: sg_smooth
  - preprocessing: sg_d1
  - preprocessing: sg_d2
  - preprocessing: absorbance_snv
  - preprocessing: absorbance_msc
  - preprocessing: absorbance_sg_smooth
  - preprocessing: absorbance_sg_d1
  - preprocessing: absorbance_sg_d2
  - preprocessing: snv_sg_smooth
  - preprocessing: snv_sg_d1
  - preprocessing: snv_sg_d2
  - preprocessing: absorbance_snv_sg_smooth
  - preprocessing: absorbance_snv_sg_d1
  - preprocessing: absorbance_snv_sg_d2

=== Matrix method: object_median ===
X shape: (394, 63)
Labels: {'almond': 214, 'peanut': 180}
  - preprocessing: raw
  - preprocessing: absorbance
  - preprocessing: snv
  - preprocessing: msc
  - preprocessing: sg_smooth
  -

,matrix_family,matrix_variant,balanced_pixel_strategy,balanced_pixel_strategy_effective,matrix_method,preprocessing,preprocessing_steps,n_observations,n_bands,n_components,m,m_effective,label_counts,evr_pc1,evr_pc2,evr_pc3,cum_pc2,cum_pc3,centroid_distance_pc1_pc2,fisher_pc1,fisher_pc2,fisher_pc3,mahalanobis_pc1_pc2,mahalanobis_pc1_pc2_pc3,ncomp_90,ncomp_95,class_trace_ratio,batch_trace_ratio,class_over_batch_ratio,train_q_mean,train_q_median,train_q_q95,train_t2_mean,train_t2_median,train_t2_q95,object_class_trace_ratio,object_batch_trace_ratio,mean_intra_object_trace,object_over_intra_ratio,n_label_almond,n_label_peanut,run_id
0,object_matrix,object_median,not_applicable,random,object_median,snv,snv,394,63,20,NaN,NaN,"{'almond': 214, 'peanut': 180}",0.396235,0.248411,0.101468,0.644647,0.746114,0.370424,0.721507,0.027104,1.392240,1.238161,2.935870,9,21,0.249102,0.054771,4.548067,0.080334,0.058969,0.213041,2.992386,2.443274,7.338692,NaN,NaN,NaN,NaN,214,180,object_matrices
1,object_matrix,object_median,not_applicable,random,object_median,msc,msc,394,63,20,NaN,NaN,"{'almond': 214, 'peanut': 180}",0.396717,0.248848,0.101294,0.645565,0.746859,0.033661,0.719933,0.026653,1.388535,1.235400,2.915389,9,21,0.248337,0.054740,4.536669,0.000662,0.000486,0.001759,2.992386,2.432507,7.371004,NaN,NaN,NaN,NaN,214,180,object_matrices
2,object_matrix,object_median,not_applicable,random,object_median,snv_sg_smooth,snv+sg_smooth,394,63,20,NaN,NaN,"{'almond': 214, 'peanut': 180}",0.452619,0.266776,0.111350,0.719395,0.830746,0.363824,0.708313,0.018647,1.407789,1.214695,2.902236,5,6,0.248807,0.055357,4.494580,0.046317,0.028271,0.147192,2.992386,2.406235,7.406666,NaN,NaN,NaN,NaN,214,180,object_matrices
3,object_matrix,object_mean,not_applicable,random,object_mean,snv_sg_smooth,snv+sg_smooth,394,63,20,NaN,NaN,"{'almond': 214, 'peanut': 180}",0.480025,0.300709,0.122042,0.780734,0.902776,0.336466,0.655774,0.280555,1.686657,1.532302,4.297894,3,4,0.303379,0.072669,4.174809,0.017629,0.011395,0.059813,2.992386,2.571584,6.785414,NaN,NaN,NaN,NaN,214,180,object_matrices
4,object_matrix,object_mean,not_applicable,random,object_mean,snv,snv,394,63,20,NaN,NaN,"{'almond': 214, 'peanut': 180}",0.461762,0.299309,0.121600,0.761071,0.882671,0.344833,0.618689,0.334197,1.679340,1.564635,4.431508,4,5,0.305118,0.072134,4.229865,0.022856,0.015998,0.073119,2.992386,2.525682,6.721754,NaN,NaN,NaN,NaN,214,180,object_matrices


PCA summary shape: (72, 42)


In [13]:
pca_scored_df = add_pca_selection_score(
    pca_summary_df,
    matrix_method_col="matrix_method",
    score_col="selection_score",
    profile="auto",
    group_col="matrix_variant",
    robust=True,
)

pca_scored_df["matrix_family"] = np.where(
    pca_scored_df["matrix_method"].isin(["object_mean", "object_median"]),
    "object_matrix",
    "pixel_matrix",
)

pca_scored_df = (
    pca_scored_df
    .sort_values(
        ["selection_score", "fisher_pc1", "class_trace_ratio"],
        ascending=False,
    )
    .reset_index(drop=True)
)

pca_scored_df.insert(0, "rank", np.arange(1, len(pca_scored_df) + 1))

display(pca_scored_df.head(N_TOP_TO_DISPLAY))

,rank,matrix_family,matrix_variant,balanced_pixel_strategy,balanced_pixel_strategy_effective,matrix_method,preprocessing,preprocessing_steps,n_observations,n_bands,n_components,m,m_effective,label_counts,evr_pc1,evr_pc2,evr_pc3,cum_pc2,cum_pc3,centroid_distance_pc1_pc2,fisher_pc1,fisher_pc2,fisher_pc3,mahalanobis_pc1_pc2,mahalanobis_pc1_pc2_pc3,ncomp_90,ncomp_95,class_trace_ratio,batch_trace_ratio,class_over_batch_ratio,train_q_mean,train_q_median,train_q_q95,train_t2_mean,train_t2_median,train_t2_q95,object_class_trace_ratio,object_batch_trace_ratio,mean_intra_object_trace,object_over_intra_ratio,n_label_almond,n_label_peanut,run_id,contrib_plus_class_trace_ratio,contrib_plus_mahalanobis_pc1_pc2_pc3,contrib_minus_batch_trace_ratio,contrib_minus_mean_train_projection_shift_norm,contrib_minus_projection_q_deviation,contrib_minus_ncomp_95,selection_score,contrib_plus_object_class_trace_ratio,contrib_plus_object_over_intra_ratio,contrib_minus_object_batch_trace_ratio,contrib_minus_mean_intra_object_trace,selection_flag
0,1,object_matrix,object_median,not_applicable,random,object_median,absorbance_sg_d1,absorbance+sg_d1,394,63,20,NaN,NaN,"{'almond': 214, 'peanut': 180}",0.686619,0.219259,0.043505,0.905878,0.949383,0.001124,0.033790,0.725992,0.001909,0.819842,0.820368,2,4,0.077787,0.016322,4.765700,2.317419e-07,2.007392e-07,5.436501e-07,2.992386,2.231761,7.832606,NaN,NaN,NaN,NaN,214,180,object_matrices,-0.854643,-0.074690,19.061813,-0.0,-0.0,0.054545,18.187026,NaN,NaN,NaN,NaN,weak_class_separation
1,2,object_matrix,object_median,not_applicable,random,object_median,absorbance_sg_d2,absorbance+sg_d2,394,63,20,NaN,NaN,"{'almond': 214, 'peanut': 180}",0.795240,0.117608,0.028355,0.912847,0.941203,0.000027,0.002126,0.512237,0.543928,0.027120,0.030267,2,4,0.033561,0.018759,1.789113,4.376039e-10,3.562546e-10,1.086301e-09,2.992386,2.216219,8.360264,NaN,NaN,NaN,NaN,214,180,object_matrices,-1.502778,-0.417778,18.878137,-0.0,-0.0,0.054545,17.012127,NaN,NaN,NaN,NaN,weak_class_separation
2,3,object_matrix,object_median,not_applicable,random,object_median,absorbance,absorbance,394,63,20,NaN,NaN,"{'almond': 214, 'peanut': 180}",0.943681,0.045486,0.004674,0.989167,0.993841,0.260488,0.079432,0.279653,0.079566,0.870078,0.990189,1,2,0.043054,0.041268,1.043270,2.538754e-03,2.167610e-03,5.072360e-03,2.992386,2.386337,7.893629,NaN,NaN,NaN,NaN,214,180,object_matrices,-1.363661,-0.000947,7.565028,-0.0,-0.0,0.272727,6.473148,NaN,NaN,NaN,NaN,weak_class_separation
3,4,object_matrix,object_median,not_applicable,random,object_median,absorbance_sg_smooth,absorbance+sg_smooth,394,63,20,NaN,NaN,"{'almond': 214, 'peanut': 180}",0.945494,0.045317,0.004519,0.990811,0.995330,0.260253,0.079403,0.279254,0.083052,0.869489,0.994553,1,2,0.043018,0.041273,1.042268,1.920438e-03,1.622925e-03,4.069492e-03,2.992386,2.387787,7.788756,NaN,NaN,NaN,NaN,214,180,object_matrices,-1.364191,0.000947,7.562541,-0.0,-0.0,0.272727,6.472025,NaN,NaN,NaN,NaN,weak_class_separation
4,5,object_matrix,object_mean,not_applicable,random,object_mean,absorbance_sg_d1,absorbance+sg_d1,394,63,20,NaN,NaN,"{'almond': 214, 'peanut': 180}",0.735538,0.186229,0.039354,0.921767,0.961121,0.000839,0.006383,0.697112,0.060923,0.679209,0.692340,2,3,0.055729,0.018513,3.010237,1.370801e-07,1.027604e-07,3.649728e-07,2.992386,2.157150,7.762081,NaN,NaN,NaN,NaN,214,180,object_matrices,-0.670633,-0.049037,5.607016,-0.0,-0.0,-0.000000,4.887345,NaN,NaN,NaN,NaN,weak_class_separation
5,6,pixel_matrix,balanced_pixels_center,center,center,balanced_pixels,absorbance_sg_d1,absorbance+sg_d1,15440,63,20,40.0,40.0,"{'almond': 8414, 'peanut': 7026}",0.977894,0.016982,0.001965,0.994876,0.996840,0.001179,0.001789,0.162706,0.018766,0.485125,0.495608,1,1,0.002093,0.000517,4.046647,5.273400e-07,3.484390e-07,1.306162e-06,2.999806,1.306581,6.561216,0.025451,0.006389,1.585349e-04,0.088611,8414,7026,balanced_pixels_center,NaN,NaN,NaN,-0.0,-0.0,0.163636,4.653973,-0.057451,-0.533859,4.573436,0.508210,weak_object_separation
6,7,pixel_matrix,bala

In [14]:
pca_scored_df.columns

Index(['rank', 'matrix_family', 'matrix_variant', 'balanced_pixel_strategy',
       'balanced_pixel_strategy_effective', 'matrix_method', 'preprocessing',
       'preprocessing_steps', 'n_observations', 'n_bands', 'n_components', 'm',
       'm_effective', 'label_counts', 'evr_pc1', 'evr_pc2', 'evr_pc3',
       'cum_pc2', 'cum_pc3', 'centroid_distance_pc1_pc2', 'fisher_pc1',
       'fisher_pc2', 'fisher_pc3', 'mahalanobis_pc1_pc2',
       'mahalanobis_pc1_pc2_pc3', 'ncomp_90', 'ncomp_95', 'class_trace_ratio',
       'batch_trace_ratio', 'class_over_batch_ratio', 'train_q_mean',
       'train_q_median', 'train_q_q95', 'train_t2_mean', 'train_t2_median',
       'train_t2_q95', 'object_class_trace_ratio', 'object_batch_trace_ratio',
       'mean_intra_object_trace', 'object_over_intra_ratio', 'n_label_almond',
       'n_label_peanut', 'run_id', 'contrib_plus_class_trace_ratio',
       'contrib_plus_mahalanobis_pc1_pc2_pc3',
       'contrib_minus_batch_trace_ratio',
       'contrib_minus_m

In [15]:
print("pca_summary_df columns:")
print(sorted(pca_summary_df.columns))

print("\npca_scored_df columns:")
print(sorted(pca_scored_df.columns))

assert "matrix_variant" in pca_summary_df.columns
assert "matrix_variant" in pca_scored_df.columns
assert pca_scored_df["matrix_variant"].notna().all()

pca_scored_df[
    ["matrix_family", "matrix_method", "balanced_pixel_strategy", "matrix_variant"]
].drop_duplicates()

pca_summary_df columns:
['balanced_pixel_strategy', 'balanced_pixel_strategy_effective', 'batch_trace_ratio', 'centroid_distance_pc1_pc2', 'class_over_batch_ratio', 'class_trace_ratio', 'cum_pc2', 'cum_pc3', 'evr_pc1', 'evr_pc2', 'evr_pc3', 'fisher_pc1', 'fisher_pc2', 'fisher_pc3', 'label_counts', 'm', 'm_effective', 'mahalanobis_pc1_pc2', 'mahalanobis_pc1_pc2_pc3', 'matrix_family', 'matrix_method', 'matrix_variant', 'mean_intra_object_trace', 'n_bands', 'n_components', 'n_label_almond', 'n_label_peanut', 'n_observations', 'ncomp_90', 'ncomp_95', 'object_batch_trace_ratio', 'object_class_trace_ratio', 'object_over_intra_ratio', 'preprocessing', 'preprocessing_steps', 'run_id', 'train_q_mean', 'train_q_median', 'train_q_q95', 'train_t2_mean', 'train_t2_median', 'train_t2_q95']

pca_scored_df columns:
['balanced_pixel_strategy', 'balanced_pixel_strategy_effective', 'batch_trace_ratio', 'centroid_distance_pc1_pc2', 'class_over_batch_ratio', 'class_trace_ratio', 'contrib_minus_batch_trace_

,matrix_family,matrix_method,balanced_pixel_strategy,matrix_variant
0,object_matrix,object_median,not_applicable,object_median
4,object_matrix,object_mean,not_applicable,object_mean
5,pixel_matrix,balanced_pixels,center,balanced_pixels_center
13,pixel_matrix,balanced_pixels,random,balanced_pixels_random


In [16]:
# The scored table is the useful PCA summary for downstream notebooks.
save_parquet(pca_scored_df, PCA_SUMMARY_PATH)

print("Saved PCA summary:")
print(" -", PCA_SUMMARY_PATH)

Saved PCA summary:
 - C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\03_pca_non_noisy_all\pca_summary.parquet


## 2. Global PCA ranking

We now inspect the best preprocessing × matrix combinations.

Useful interpretation:

- High class separation is good.
- High batch effect is undesirable.
- For pixel matrices, object-level separation is more important than raw pixel-level separation.
- The final choice should not rely on one metric only.

In [17]:
ranking_cols = [
    "rank",
    "matrix_family",
    "matrix_variant",
    "matrix_method",
    "balanced_pixel_strategy",
    "preprocessing",
    "selection_score",
    "selection_flag",
    "n_observations",
    "n_label_almond",
    "n_label_peanut",
    "label_counts",
    "evr_pc1",
    "cum_pc3",
    "ncomp_90",
    "ncomp_95",
    "fisher_pc1",
    "fisher_pc2",
    "mahalanobis_pc1_pc2_pc3",
    "class_trace_ratio",
    "batch_trace_ratio",
    "class_over_batch_ratio",
    "object_class_trace_ratio",
    "object_batch_trace_ratio",
    "object_over_intra_ratio",
    "mean_intra_object_trace",
]

top_candidates_df = pca_scored_df.copy()
#top_candidates_df.insert(0, "rank", np.arange(1, len(top_candidates_df) + 1))

available_ranking_cols = [col for col in ranking_cols if col in top_candidates_df.columns]

top_candidates_df[available_ranking_cols].head(N_TOP_TO_DISPLAY)

,rank,matrix_family,matrix_variant,matrix_method,balanced_pixel_strategy,preprocessing,selection_score,selection_flag,n_observations,n_label_almond,n_label_peanut,label_counts,evr_pc1,cum_pc3,ncomp_90,ncomp_95,fisher_pc1,fisher_pc2,mahalanobis_pc1_pc2_pc3,class_trace_ratio,batch_trace_ratio,class_over_batch_ratio,object_class_trace_ratio,object_batch_trace_ratio,object_over_intra_ratio,mean_intra_object_trace
0,1,object_matrix,object_median,object_median,not_applicable,absorbance_sg_d1,18.187026,weak_class_separation,394,214,180,"{'almond': 214, 'peanut': 180}",0.686619,0.949383,2,4,0.033790,0.725992,0.820368,0.077787,0.016322,4.765700,NaN,NaN,NaN,NaN
1,2,object_matrix,object_median,object_median,not_applicable,absorbance_sg_d2,17.012127,weak_class_separation,394,214,180,"{'almond': 214, 'peanut': 180}",0.795240,0.941203,2,4,0.002126,0.512237,0.030267,0.033561,0.018759,1.789113,NaN,NaN,NaN,NaN
2,3,object_matrix,object_median,object_median,not_applicable,absorbance,6.473148,weak_class_separation,394,214,180,"{'almond': 214, 'peanut': 180}",0.943681,0.993841,1,2,0.079432,0.279653,0.990189,0.043054,0.041268,1.043270,NaN,NaN,NaN,NaN
3,4,object_matrix,object_median,object_median,not_applicable,absorbance_sg_smooth,6.472025,weak_class_separation,394,214,180,"{'almond': 214, 'peanut': 180}",0.945494,0.995330,1,2,0.079403,0.279254,0.994553,0.043018,0.041273,1.042268,NaN,NaN,NaN,NaN
4,5,object_matrix,object_mean,object_mean,not_applicable,absorbance_sg_d1,4.887345,weak_class_separation,394,214,180,"{'almond': 214, 'peanut': 180}",0.735538,0.961121,2,3,0.006383,0.697112,0.692340,0.055729,0.018513,3.010237,NaN,NaN,NaN,NaN
5,6,pixel_matrix,balanced_pixels_center,balanced_pixels,center,absorbance_sg_d1,4.653973,weak_object_separation,15440,8414,7026,"{'almond': 8414, 'peanut': 7026}",0.977894,0.996840,1,1,0.001789,0.162706,0.495608,0.002093,0.000517,4.046647,0.025451,0.006389,0.088611,1.585349e-04
6,7,pixel_matrix,balanced_pixels_center,balanced_pixels,center,absorbance_sg_d2,4.366398,weak_object_separation,15440,8414,7026,"{'almond': 8414, 'peanut': 7026}",0.996527,0.999187,1,1,0.001992,0.069682,0.046046,0.001006,0.000154,6.537100,0.017817,0.002848,0.058737,9.306022e-07
7,8,object_matrix,object_mean,object_mean,not_applicable,absorbance_sg_d2,4.314959,weak_class_separation,394,214,180,"{'almond': 214, 'peanut': 180}",0.846272,0.971727,2,3,0.000026,0.427958,0.023053,0.027787,0.022875,1.214738,NaN,NaN,NaN,NaN
8,9,pixel_matrix,balanced_pixels_center,balanced_pixels,center,sg_smooth,3.312388,weak_object_separation,15440,8414,7026,"{'almond': 8414, 'peanut': 7026}",0.960334,0.997270,1,1,0.018388,0.085288,0.516170,0.010264,0.039741,0.258284,0.017376,0.080380,1.188804,3.530972e-01
9,10,object_matrix,object_mean,object_mean,not_applicable,absorbance_snv_sg_smooth,3.307124,batch_sensitive,394,214,180,"{'almond': 214, 'peanut': 180}",0.595743,0.917383,3,4,0.520939,1.553504,3.988180,0.343847,0.068605,5.011950,NaN,NaN,NaN,NaN


In [18]:
top_candidates_objects_df = top_candidates_df[top_candidates_df['matrix_method'].str.startswith('object')]
top_candidates_objects_df[available_ranking_cols]

,rank,matrix_family,matrix_variant,matrix_method,balanced_pixel_strategy,preprocessing,selection_score,selection_flag,n_observations,n_label_almond,n_label_peanut,label_counts,evr_pc1,cum_pc3,ncomp_90,ncomp_95,fisher_pc1,fisher_pc2,mahalanobis_pc1_pc2_pc3,class_trace_ratio,batch_trace_ratio,class_over_batch_ratio,object_class_trace_ratio,object_batch_trace_ratio,object_over_intra_ratio,mean_intra_object_trace
0,1,object_matrix,object_median,object_median,not_applicable,absorbance_sg_d1,18.187026,weak_class_separation,394,214,180,"{'almond': 214, 'peanut': 180}",0.686619,0.949383,2,4,0.033790,0.725992,0.820368,0.077787,0.016322,4.765700,NaN,NaN,NaN,NaN
1,2,object_matrix,object_median,object_median,not_applicable,absorbance_sg_d2,17.012127,weak_class_separation,394,214,180,"{'almond': 214, 'peanut': 180}",0.795240,0.941203,2,4,0.002126,0.512237,0.030267,0.033561,0.018759,1.789113,NaN,NaN,NaN,NaN
2,3,object_matrix,object_median,object_median,not_applicable,absorbance,6.473148,weak_class_separation,394,214,180,"{'almond': 214, 'peanut': 180}",0.943681,0.993841,1,2,0.079432,0.279653,0.990189,0.043054,0.041268,1.043270,NaN,NaN,NaN,NaN
3,4,object_matrix,object_median,object_median,not_applicable,absorbance_sg_smooth,6.472025,weak_class_separation,394,214,180,"{'almond': 214, 'peanut': 180}",0.945494,0.995330,1,2,0.079403,0.279254,0.994553,0.043018,0.041273,1.042268,NaN,NaN,NaN,NaN
4,5,object_matrix,object_mean,object_mean,not_applicable,absorbance_sg_d1,4.887345,weak_class_separation,394,214,180,"{'almond': 214, 'peanut': 180}",0.735538,0.961121,2,3,0.006383,0.697112,0.692340,0.055729,0.018513,3.010237,NaN,NaN,NaN,NaN
7,8,object_matrix,object_mean,object_mean,not_applicable,absorbance_sg_d2,4.314959,weak_class_separation,394,214,180,"{'almond': 214, 'peanut': 180}",0.846272,0.971727,2,3,0.000026,0.427958,0.023053,0.027787,0.022875,1.214738,NaN,NaN,NaN,NaN
9,10,object_matrix,object_mean,object_mean,not_applicable,absorbance_snv_sg_smooth,3.307124,batch_sensitive,394,214,180,"{'almond': 214, 'peanut': 180}",0.595743,0.917383,3,4,0.520939,1.553504,3.988180,0.343847,0.068605,5.011950,NaN,NaN,NaN,NaN
11,12,object_matrix,object_mean,object_mean,not_applicable,absorbance_msc,3.235770,batch_sensitive,394,214,180,"{'almond': 214, 'peanut': 180}",0.571238,0.893044,4,5,0.520037,1.513861,4.159365,0.346190,0.067272,5.146128,NaN,NaN,NaN,NaN
12,13,object_matrix,object_mean,object_mean,not_applicable,absorbance_snv,3.232337,batch_sensitive,394,214,180,"{'almond': 214, 'peanut': 180}",0.570045,0.892202,4,5,0.521457,1.518728,4.179036,0.347045,0.067360,5.152086,NaN,NaN,NaN,NaN
17,18,object_matrix,object_median,object_median,not_applicable,snv_sg_smooth,2.801495,batch_sensitive,394,214,180,"{'almond': 214, 'peanut': 180}",0.452619,0.830746,5,6,0.708313,0.018647,2.902236,0.248807,0.055357,4.494580,NaN,NaN,NaN,NaN


In [19]:
top_candidates_pixels_df = top_candidates_df[~top_candidates_df['matrix_method'].str.startswith('object')]
top_candidates_pixels_df[available_ranking_cols]

,rank,matrix_family,matrix_variant,matrix_method,balanced_pixel_strategy,preprocessing,selection_score,selection_flag,n_observations,n_label_almond,n_label_peanut,label_counts,evr_pc1,cum_pc3,ncomp_90,ncomp_95,fisher_pc1,fisher_pc2,mahalanobis_pc1_pc2_pc3,class_trace_ratio,batch_trace_ratio,class_over_batch_ratio,object_class_trace_ratio,object_batch_trace_ratio,object_over_intra_ratio,mean_intra_object_trace
5,6,pixel_matrix,balanced_pixels_center,balanced_pixels,center,absorbance_sg_d1,4.653973,weak_object_separation,15440,8414,7026,"{'almond': 8414, 'peanut': 7026}",0.977894,0.996840,1,1,0.001789,0.162706,0.495608,0.002093,0.000517,4.046647,0.025451,0.006389,0.088611,1.585349e-04
6,7,pixel_matrix,balanced_pixels_center,balanced_pixels,center,absorbance_sg_d2,4.366398,weak_object_separation,15440,8414,7026,"{'almond': 8414, 'peanut': 7026}",0.996527,0.999187,1,1,0.001992,0.069682,0.046046,0.001006,0.000154,6.537100,0.017817,0.002848,0.058737,9.306022e-07
8,9,pixel_matrix,balanced_pixels_center,balanced_pixels,center,sg_smooth,3.312388,weak_object_separation,15440,8414,7026,"{'almond': 8414, 'peanut': 7026}",0.960334,0.997270,1,1,0.018388,0.085288,0.516170,0.010264,0.039741,0.258284,0.017376,0.080380,1.188804,3.530972e-01
10,11,pixel_matrix,balanced_pixels_center,balanced_pixels,center,raw,3.306730,weak_object_separation,15440,8414,7026,"{'almond': 8414, 'peanut': 7026}",0.958876,0.996152,1,1,0.018369,0.084764,0.516337,0.010256,0.039796,0.257714,0.017358,0.080476,1.189325,3.531417e-01
13,14,pixel_matrix,balanced_pixels_random,balanced_pixels,random,snv_sg_d1,3.095589,batch_sensitive,15440,8414,7026,"{'almond': 8414, 'peanut': 7026}",0.809471,0.948605,2,4,0.041120,0.062526,0.482433,0.020718,0.015327,1.351694,0.115447,0.082807,0.246077,4.465343e-04
14,15,pixel_matrix,balanced_pixels_random,balanced_pixels,random,snv_sg_smooth,3.010389,batch_sensitive,15440,8414,7026,"{'almond': 8414, 'peanut': 7026}",0.647148,0.852468,4,5,0.012200,0.192899,0.675574,0.017588,0.008022,2.192394,0.166178,0.069824,0.139294,1.088021e+00
15,16,pixel_matrix,balanced_pixels_center,balanced_pixels,center,snv_sg_smooth,2.939204,batch_sensitive,15440,8414,7026,"{'almond': 8414, 'peanut': 7026}",0.560744,0.836021,4,5,0.048368,0.218433,0.795851,0.036484,0.024759,1.473579,0.142381,0.094443,0.382495,4.970738e-01
16,17,pixel_matrix,balanced_pixels_random,balanced_pixels,random,absorbance_snv_sg_smooth,2.896872,batch_sensitive,15440,8414,7026,"{'almond': 8414, 'peanut': 7026}",0.531770,0.866298,4,5,0.020412,0.071153,0.492231,0.017266,0.008086,2.135170,0.154246,0.067487,0.144812,1.303934e+00
20,21,pixel_matrix,balanced_pixels_random,balanced_pixels,random,absorbance_snv_sg_d1,2.655268,batch_sensitive,15440,8414,7026,"{'almond': 8414, 'peanut': 7026}",0.850470,0.965273,2,3,0.037194,0.027923,0.389575,0.017636,0.013911,1.267767,0.113414,0.087241,0.206564,7.714657e-04
22,23,pixel_matrix,balanced_pixels_random,balanced_pixels,random,snv,2.508121,batch_sensitive,15440,8414,7026,"{'almond': 8414, 'peanut': 7026}",0.585212,0.784416,5,11,0.012346,0.165550,0.609890,0.015900,0.007870,2.020398,0.147859,0.068284,0.139341,1.113244e+00


In [20]:
plot_pca_metric_ranking(
    top_candidates_objects_df,
    metric="selection_score",
    group_col="matrix_method",
    label_col="preprocessing",
    ascending=False,
    top_n=20,
    title="PCA selection score ranking by matrix representation",
    show=True,
)

In [21]:
plot_pca_metric_ranking(
    top_candidates_pixels_df,
    metric="selection_score",
    group_col="matrix_variant",
    label_col="preprocessing",
    ascending=False,
    top_n=20,
    title="PCA selection score ranking by matrix representation",
    show=True,
)

In [22]:
plot_pca_metric_heatmap(
    pca_scored_df,
    metric="class_trace_ratio",
    index_col="preprocessing",
    column_col="matrix_variant",
    title="PCA class trace ratio — preprocessing × matrix representation",
    show=True,
)

In [23]:
plot_pca_metric_heatmap(
    pca_scored_df,
    metric="batch_trace_ratio",
    index_col="preprocessing",
    column_col="matrix_variant",
    title="PCA batch trace ratio — preprocessing × matrix representation",
    show=True,
)

In [24]:
plot_pca_metric_tradeoff(
    pca_scored_df,
    x_metric="batch_trace_ratio",
    y_metric="class_trace_ratio",
    color_by="matrix_variant",
    symbol_by=None,
    show_pareto=True,
    label_col="preprocessing",
    label_top_n=8,
    title="PCA trade-off — class separation vs batch effect",
    show=True,
)

In [25]:
pixel_metric_df = pca_scored_df[
    pca_scored_df["matrix_method"].isin(["balanced_pixels", "all_pixels"])
].copy()

if len(pixel_metric_df) > 0:
    plot_pca_metric_tradeoff(
        pixel_metric_df,
        x_metric="object_batch_trace_ratio",
        y_metric="object_class_trace_ratio",
        color_by="matrix_variant",
        symbol_by=None,
        show_pareto=True,
        label_col="preprocessing",
        label_top_n=8,
        size_by="object_over_intra_ratio",
        title="Pixel matrix trade-off — object class separation vs object batch effect",
        show=True,
    )
else:
    print("No pixel-level matrix result available.")

In [26]:
def get_pca_result_from_row(row, registry):
    """
    Retrieve full PCA result from one summary row.
    """
    run_id = row["run_id"]
    matrix_method = row["matrix_method"]
    preprocessing = row["preprocessing"]

    return registry[run_id][matrix_method][preprocessing]


def metadata_value(metadata, *possible_keys, default=None):
    """
    Robustly retrieve one metadata array from possible keys.
    """
    for key in possible_keys:
        if key in metadata and metadata[key] is not None:
            return metadata[key]
    return default

In [27]:
best_by_variant_pixels_df = top_candidates_pixels_df.sort_values("selection_score", ascending=False).groupby("matrix_variant", as_index=False).first().reset_index(drop=True)
best_by_variant_pixels_df

,matrix_variant,rank,matrix_family,balanced_pixel_strategy,balanced_pixel_strategy_effective,matrix_method,preprocessing,preprocessing_steps,n_observations,n_bands,n_components,m,m_effective,label_counts,evr_pc1,evr_pc2,evr_pc3,cum_pc2,cum_pc3,centroid_distance_pc1_pc2,fisher_pc1,fisher_pc2,fisher_pc3,mahalanobis_pc1_pc2,mahalanobis_pc1_pc2_pc3,ncomp_90,ncomp_95,class_trace_ratio,batch_trace_ratio,class_over_batch_ratio,train_q_mean,train_q_median,train_q_q95,train_t2_mean,train_t2_median,train_t2_q95,object_class_trace_ratio,object_batch_trace_ratio,mean_intra_object_trace,object_over_intra_ratio,n_label_almond,n_label_peanut,run_id,contrib_plus_class_trace_ratio,contrib_plus_mahalanobis_pc1_pc2_pc3,contrib_minus_batch_trace_ratio,contrib_minus_mean_train_projection_shift_norm,contrib_minus_projection_q_deviation,contrib_minus_ncomp_95,selection_score,contrib_plus_object_class_trace_ratio,contrib_plus_object_over_intra_ratio,contrib_minus_object_batch_trace_ratio,contrib_minus_mean_intra_object_trace,selection_flag
0,balanced_pixels_center,6,pixel_matrix,center,center,balanced_pixels,absorbance_sg_d1,absorbance+sg_d1,15440,63,20,40.0,40.0,"{'almond': 8414, 'peanut': 7026}",0.977894,0.016982,0.001965,0.994876,0.996840,0.001179,0.001789,0.162706,0.018766,0.485125,0.495608,1,1,0.002093,0.000517,4.046647,5.273400e-07,3.484390e-07,0.000001,2.999806,1.306581,6.561216,0.025451,0.006389,0.000159,0.088611,8414,7026,balanced_pixels_center,NaN,NaN,NaN,-0.0,-0.0,0.163636,4.653973,-0.057451,-0.533859,4.573436,0.508210,weak_object_separation
1,balanced_pixels_random,14,pixel_matrix,random,random,balanced_pixels,snv_sg_d1,snv+sg_d1,15440,63,20,40.0,40.0,"{'almond': 8414, 'peanut': 7026}",0.809471,0.093495,0.045640,0.902966,0.948605,0.006588,0.041120,0.062526,0.010598,0.458101,0.482433,2,4,0.020718,0.015327,1.351694,2.909131e-05,1.784016e-05,0.000070,2.999806,1.548295,10.284589,0.115447,0.082807,0.000447,0.246077,8414,7026,balanced_pixels_random,NaN,NaN,NaN,-0.0,-0.0,-0.109091,3.095589,2.538048,0.875886,-0.628001,0.418747,batch_sensitive


In [28]:
best_by_variant_objects_df = top_candidates_objects_df.sort_values("selection_score", ascending=False).groupby("matrix_variant", as_index=False).first().reset_index(drop=True)
best_by_variant_objects_df

,matrix_variant,rank,matrix_family,balanced_pixel_strategy,balanced_pixel_strategy_effective,matrix_method,preprocessing,preprocessing_steps,n_observations,n_bands,n_components,m,m_effective,label_counts,evr_pc1,evr_pc2,evr_pc3,cum_pc2,cum_pc3,centroid_distance_pc1_pc2,fisher_pc1,fisher_pc2,fisher_pc3,mahalanobis_pc1_pc2,mahalanobis_pc1_pc2_pc3,ncomp_90,ncomp_95,class_trace_ratio,batch_trace_ratio,class_over_batch_ratio,train_q_mean,train_q_median,train_q_q95,train_t2_mean,train_t2_median,train_t2_q95,object_class_trace_ratio,object_batch_trace_ratio,mean_intra_object_trace,object_over_intra_ratio,n_label_almond,n_label_peanut,run_id,contrib_plus_class_trace_ratio,contrib_plus_mahalanobis_pc1_pc2_pc3,contrib_minus_batch_trace_ratio,contrib_minus_mean_train_projection_shift_norm,contrib_minus_projection_q_deviation,contrib_minus_ncomp_95,selection_score,contrib_plus_object_class_trace_ratio,contrib_plus_object_over_intra_ratio,contrib_minus_object_batch_trace_ratio,contrib_minus_mean_intra_object_trace,selection_flag
0,object_mean,5,object_matrix,not_applicable,random,object_mean,absorbance_sg_d1,absorbance+sg_d1,394,63,20,NaN,NaN,"{'almond': 214, 'peanut': 180}",0.735538,0.186229,0.039354,0.921767,0.961121,0.000839,0.006383,0.697112,0.060923,0.679209,0.692340,2,3,0.055729,0.018513,3.010237,1.370801e-07,1.027604e-07,3.649728e-07,2.992386,2.157150,7.762081,NaN,NaN,NaN,NaN,214,180,object_matrices,-0.670633,-0.049037,5.607016,-0.0,-0.0,-0.000000,4.887345,NaN,NaN,NaN,NaN,weak_class_separation
1,object_median,1,object_matrix,not_applicable,random,object_median,absorbance_sg_d1,absorbance+sg_d1,394,63,20,NaN,NaN,"{'almond': 214, 'peanut': 180}",0.686619,0.219259,0.043505,0.905878,0.949383,0.001124,0.033790,0.725992,0.001909,0.819842,0.820368,2,4,0.077787,0.016322,4.765700,2.317419e-07,2.007392e-07,5.436501e-07,2.992386,2.231761,7.832606,NaN,NaN,NaN,NaN,214,180,object_matrices,-0.854643,-0.074690,19.061813,-0.0,-0.0,0.054545,18.187026,NaN,NaN,NaN,NaN,weak_class_separation


In [29]:
best_by_variant_df = pd.concat([best_by_variant_objects_df, best_by_variant_pixels_df], ignore_index=True)
best_by_variant_df

,matrix_variant,rank,matrix_family,balanced_pixel_strategy,balanced_pixel_strategy_effective,matrix_method,preprocessing,preprocessing_steps,n_observations,n_bands,n_components,m,m_effective,label_counts,evr_pc1,evr_pc2,evr_pc3,cum_pc2,cum_pc3,centroid_distance_pc1_pc2,fisher_pc1,fisher_pc2,fisher_pc3,mahalanobis_pc1_pc2,mahalanobis_pc1_pc2_pc3,ncomp_90,ncomp_95,class_trace_ratio,batch_trace_ratio,class_over_batch_ratio,train_q_mean,train_q_median,train_q_q95,train_t2_mean,train_t2_median,train_t2_q95,object_class_trace_ratio,object_batch_trace_ratio,mean_intra_object_trace,object_over_intra_ratio,n_label_almond,n_label_peanut,run_id,contrib_plus_class_trace_ratio,contrib_plus_mahalanobis_pc1_pc2_pc3,contrib_minus_batch_trace_ratio,contrib_minus_mean_train_projection_shift_norm,contrib_minus_projection_q_deviation,contrib_minus_ncomp_95,selection_score,contrib_plus_object_class_trace_ratio,contrib_plus_object_over_intra_ratio,contrib_minus_object_batch_trace_ratio,contrib_minus_mean_intra_object_trace,selection_flag
0,object_mean,5,object_matrix,not_applicable,random,object_mean,absorbance_sg_d1,absorbance+sg_d1,394,63,20,NaN,NaN,"{'almond': 214, 'peanut': 180}",0.735538,0.186229,0.039354,0.921767,0.961121,0.000839,0.006383,0.697112,0.060923,0.679209,0.692340,2,3,0.055729,0.018513,3.010237,1.370801e-07,1.027604e-07,3.649728e-07,2.992386,2.157150,7.762081,NaN,NaN,NaN,NaN,214,180,object_matrices,-0.670633,-0.049037,5.607016,-0.0,-0.0,-0.000000,4.887345,NaN,NaN,NaN,NaN,weak_class_separation
1,object_median,1,object_matrix,not_applicable,random,object_median,absorbance_sg_d1,absorbance+sg_d1,394,63,20,NaN,NaN,"{'almond': 214, 'peanut': 180}",0.686619,0.219259,0.043505,0.905878,0.949383,0.001124,0.033790,0.725992,0.001909,0.819842,0.820368,2,4,0.077787,0.016322,4.765700,2.317419e-07,2.007392e-07,5.436501e-07,2.992386,2.231761,7.832606,NaN,NaN,NaN,NaN,214,180,object_matrices,-0.854643,-0.074690,19.061813,-0.0,-0.0,0.054545,18.187026,NaN,NaN,NaN,NaN,weak_class_separation
2,balanced_pixels_center,6,pixel_matrix,center,center,balanced_pixels,absorbance_sg_d1,absorbance+sg_d1,15440,63,20,40.0,40.0,"{'almond': 8414, 'peanut': 7026}",0.977894,0.016982,0.001965,0.994876,0.996840,0.001179,0.001789,0.162706,0.018766,0.485125,0.495608,1,1,0.002093,0.000517,4.046647,5.273400e-07,3.484390e-07,1.306162e-06,2.999806,1.306581,6.561216,0.025451,0.006389,0.000159,0.088611,8414,7026,balanced_pixels_center,NaN,NaN,NaN,-0.0,-0.0,0.163636,4.653973,-0.057451,-0.533859,4.573436,0.508210,weak_object_separation
3,balanced_pixels_random,14,pixel_matrix,random,random,balanced_pixels,snv_sg_d1,snv+sg_d1,15440,63,20,40.0,40.0,"{'almond': 8414, 'peanut': 7026}",0.809471,0.093495,0.045640,0.902966,0.948605,0.006588,0.041120,0.062526,0.010598,0.458101,0.482433,2,4,0.020718,0.015327,1.351694,2.909131e-05,1.784016e-05,7.015957e-05,2.999806,1.548295,10.284589,0.115447,0.082807,0.000447,0.246077,8414,7026,balanced_pixels_random,NaN,NaN,NaN,-0.0,-0.0,-0.109091,3.095589,2.538048,0.875886,-0.628001,0.418747,batch_sensitive


In [30]:
# Detailed plots: top candidate globally + best candidate for each matrix variant.
selected_rows = []

selected_rows.append(pca_scored_df.iloc[0])

for _, row in best_by_variant_df.iterrows():
    selected_rows.append(row)

# Deduplicate by run_id + matrix_method + preprocessing
selected_df = pd.DataFrame(selected_rows).drop_duplicates(
    subset=["run_id", "matrix_method", "preprocessing", "balanced_pixel_strategy"]
).reset_index(drop=True)

selected_df = selected_df.head(N_TOP_TO_PLOT)

display(
    selected_df[
        [
            "run_id",
            "matrix_variant",
            "matrix_method",
            "balanced_pixel_strategy",
            "preprocessing",
            "selection_score",
            "selection_flag",
            "class_trace_ratio",
            "batch_trace_ratio",
            "object_class_trace_ratio",
            "object_batch_trace_ratio",
        ]
    ]
)

,run_id,matrix_variant,matrix_method,balanced_pixel_strategy,preprocessing,selection_score,selection_flag,class_trace_ratio,batch_trace_ratio,object_class_trace_ratio,object_batch_trace_ratio
0,object_matrices,object_median,object_median,not_applicable,absorbance_sg_d1,18.187026,weak_class_separation,0.077787,0.016322,NaN,NaN
1,object_matrices,object_mean,object_mean,not_applicable,absorbance_sg_d1,4.887345,weak_class_separation,0.055729,0.018513,NaN,NaN
2,balanced_pixels_center,balanced_pixels_center,balanced_pixels,center,absorbance_sg_d1,4.653973,weak_object_separation,0.002093,0.000517,0.025451,0.006389
3,balanced_pixels_random,balanced_pixels_random,balanced_pixels,random,snv_sg_d1,3.095589,batch_sensitive,0.020718,0.015327,0.115447,0.082807


In [31]:
if RUN_DETAILED_PCA_PLOTS:
    for _, row in selected_df.iterrows():
        result = get_pca_result_from_row(row, pca_results_registry)
        pca = result["pca"]

        title = (
            f"Explained variance — {row['matrix_variant']} — "
            f"{row['preprocessing']}"
        )

        plot_explained_variance(
            pca.explained_variance_ratio_,
            pca.cumulative_explained_variance_ratio_,
            n_components_to_show=min(N_COMPONENTS, 12),
            title=title,
            show=True,
        )
else:
    print("Detailed explained variance plots skipped.")

In [32]:
if RUN_DETAILED_PCA_PLOTS:
    for _, row in selected_df.iterrows():
        result = get_pca_result_from_row(
            row,
            pca_results_registry,
        )

        pca = result["pca"]
        scores = result["scores"]
        y = result["y"]
        metadata = result["metadata"]

        matrix_method = str(row["matrix_method"])
        is_pixel_matrix = matrix_method in {
            "balanced_pixels",
            "all_pixels",
        }

        title = (
            f"PCA scores PC1/PC2 — "
            f"{row['matrix_variant']} — "
            f"{row['preprocessing']}"
        )

        if not is_pixel_matrix:
            plot_scores(
                scores,
                dims=(1, 2),
                labels=y,
                color_by="label",
                object_ids=metadata_value(
                    metadata,
                    "object_id",
                    "observation_ids",
                ),
                source_images=metadata_value(
                    metadata,
                    "source_image",
                    "source_images",
                ),
                batches=metadata_value(
                    metadata,
                    "batch",
                    "batches",
                ),
                areas=metadata_value(
                    metadata,
                    "area_pixels",
                    "area",
                    "areas",
                ),
                symbol_by="batch",
                category_order=["almond", "peanut"],
                component_variance=pca.explained_variance_ratio_,
                component_prefix="PC",
                title=title,
                show=True,
            )

        else:
            score_df = build_scores_dataframe(
                scores,
                labels=y,
                meta=metadata,
                dims=(1, 2),
                score_prefix="PC",
            )

            plot_scores_density(
                score_df,
                x="PC1",
                y="PC2",
                color_by="label",
                facet_col=(
                    "batch"
                    if "batch" in score_df.columns
                    else None
                ),
                mode="contour",
                title=title,
                show=True,
            )

In [34]:
if RUN_DETAILED_PCA_PLOTS:
    for _, row in selected_df.iterrows():
        result = get_pca_result_from_row(
            row,
            pca_results_registry,
        )

        pca = result["pca"]
        scores = result["scores"]
        y = result["y"]
        metadata = result["metadata"]

        matrix_method = str(row["matrix_method"])
        is_pixel_matrix = matrix_method in {
            "balanced_pixels",
            "all_pixels",
        }

        title = (
            f"PCA scores PC1/PC2 colored by batch — {row['matrix_variant']} — "
            f"{row['preprocessing']}"
        )

        if not is_pixel_matrix:
            plot_scores(
                scores,
                dims=(1, 2),
                labels=y,
                color_values=metadata_value(
                    metadata,
                    "batch",
                    "batches",
                ),
                color_by="batch",
                object_ids=metadata_value(
                    metadata,
                    "object_id",
                    "observation_ids",
                ),
                source_images=metadata_value(
                    metadata,
                    "source_image",
                    "source_images",
                ),
                batches=metadata_value(
                    metadata,
                    "batch",
                    "batches",
                ),
                areas=metadata_value(
                    metadata,
                    "area_pixels",
                    "area",
                    "areas",
                ),
                symbol_by="batch",
                category_order=["almond", "peanut"],
                component_variance=pca.explained_variance_ratio_,
                component_prefix="PC",
                title=title,
                show=True,
            )
        else:
            score_df = build_scores_dataframe(
                scores,
                labels=y,
                meta=metadata,
                dims=(1, 2),
                score_prefix="PC",
            )

            object_score_df = summarize_scores_by_object(
                score_df,
                score_cols=("PC1", "PC2"),
                object_col="object_id",
                extra_group_cols=[
                    "label",
                    "batch",
                    "source_image",
                    "subset",
                ],
            )

            plot_object_score_summary(
                object_score_df,
                x="PC1_mean",
                y="PC2_mean",
                color_by="label",
                symbol_by="batch",
                facet_col=None,
                size_by="n_pixels",
                title=(
                    f"Object summary of pixel PCA scores — "
                    f"{row['matrix_variant']} — "
                    f"{row['preprocessing']}"
                ),
                show=True,
            )

In [36]:
if RUN_DETAILED_PCA_PLOTS:
    for _, row in selected_df.iterrows():
        result = get_pca_result_from_row(row, pca_results_registry)

        loadings = result["loadings"]

        title = (
            f"PCA loadings — {row['matrix_variant']} — "
            f"{row['preprocessing']}"
        )

        plot_loadings(
            loadings,
            wavelengths=wavelengths,
            components=(1, 2, 3),
            explained_variance_ratio=(
                result["pca"].explained_variance_ratio_
            ),
            title=title,
            show=True,
        )
else:
    print("Detailed PCA loadings plots skipped.")

In [37]:
if RUN_DETAILED_PCA_PLOTS:
    for _, row in selected_df.iterrows():
        result = get_pca_result_from_row(row, pca_results_registry)

        pca = result["pca"]
        X_pre = result["X_preprocessed"]
        y = result["y"]
        metadata = result["metadata"]

        object_ids = metadata_value(metadata, "object_id", "observation_ids")
        source_images = metadata_value(metadata, "source_image", "source_images")

        title = (
            f"PCA diagnostic Q vs T² — {row['matrix_variant']} — "
            f"{row['preprocessing']}"
        )

        plot_pca_diagnostic(
            pca,
            X=X_pre,
            labels=y,
            object_ids=object_ids,
            source_images=source_images,
            n_components=min(3, N_COMPONENTS),
            title=title,
            show=True,
        )
else:
    print("Detailed PCA diagnostic plots skipped.")

In [38]:
def mean_class_difference_by_preprocessing(
    pca_results_registry,
    target_class="peanut",
    reference_class="almond",
):
    rows = []

    for run_id, results_by_matrix in pca_results_registry.items():
        for matrix_method, results_by_preproc in results_by_matrix.items():
            for preprocessing, res in results_by_preproc.items():
                X = res["X_preprocessed"]
                y = np.asarray(res["y"]).astype(str)

                if target_class not in y or reference_class not in y:
                    continue

                mean_target = X[y == target_class].mean(axis=0)
                mean_reference = X[y == reference_class].mean(axis=0)
                diff = mean_target - mean_reference

                rows.append({
                    "run_id": run_id,
                    "matrix_method": matrix_method,
                    "preprocessing": preprocessing,
                    "mean_abs_difference": float(np.mean(np.abs(diff))),
                    "max_abs_difference": float(np.max(np.abs(diff))),
                    "wavelength_max_difference": float(wavelengths[np.argmax(np.abs(diff))]),
                })

    return pd.DataFrame(rows)


class_difference_df = mean_class_difference_by_preprocessing(
    pca_results_registry,
    target_class=TARGET_CLASS,
    reference_class="almond",
)

display(
    class_difference_df
    .sort_values("mean_abs_difference", ascending=False)
    .head(20)
)

,run_id,matrix_method,preprocessing,mean_abs_difference,max_abs_difference,wavelength_max_difference
25,object_matrices,object_median,absorbance_snv,0.041804,0.295978,1702.000000
33,object_matrices,object_median,absorbance_snv_sg_smooth,0.041280,0.283528,1702.000000
7,object_matrices,object_mean,absorbance_snv,0.039908,0.236609,1702.000000
15,object_matrices,object_mean,absorbance_snv_sg_smooth,0.039648,0.232342,1702.000000
61,balanced_pixels_center,balanced_pixels,absorbance_snv,0.039294,0.212528,1702.000000
69,balanced_pixels_center,balanced_pixels,absorbance_snv_sg_smooth,0.039062,0.213739,1702.000000
43,balanced_pixels_random,balanced_pixels,absorbance_snv,0.038373,0.208589,1702.000000
51,balanced_pixels_random,balanced_pixels,absorbance_snv_sg_smooth,0.038274,0.207778,1702.000000
30,object_matrices,object_median,snv_sg_smooth,0.037309,0.227192,1702.000000
20,object_matrices,object_median,snv,0.037156,0.230296,1702.000000


## 3. Interpretation table

We prepare a compact interpretation table to help select candidate preprocessing methods.

For PCA/MCR exploration, good candidates should generally have:

- strong class separation,
- limited batch effect,
- interpretable loadings,
- stable spectra after preprocessing,
- no excessive loss of spectral structure.

In [39]:
interpretation_cols = [
    "matrix_variant",
    "preprocessing",
    "selection_score",
    "selection_flag",
    "n_observations",
    "ncomp_90",
    "ncomp_95",
    "cum_pc3",
    "class_trace_ratio",
    "batch_trace_ratio",
    "class_over_batch_ratio",
    "fisher_pc1",
    "fisher_pc2",
    "mahalanobis_pc1_pc2_pc3",
    "object_class_trace_ratio",
    "object_batch_trace_ratio",
    "object_over_intra_ratio",
    "mean_intra_object_trace",
]

available_cols = [col for col in interpretation_cols if col in pca_scored_df.columns]

# interpretation_df = pca_scored_df[available_cols].copy()

# interpretation_df

In [40]:
# interpretation_objects_df = interpretation_df[interpretation_df['matrix_variant'].str.startswith('object')]
# interpretation_objects_df

In [41]:
# interpretation_pixels_df = interpretation_df[~interpretation_df['matrix_variant'].str.startswith('object')]
# interpretation_pixels_df

In [42]:
# preproc_objects = interpretation_objects_df[interpretation_objects_df['selection_score']>2.5]['preprocessing'].unique().tolist()
# preproc_objects

In [43]:
# preproc_pixels = interpretation_pixels_df[interpretation_pixels_df['selection_score']>3]['preprocessing'].unique().tolist()
# preproc_pixels

In [44]:
# preproc_study = set(preproc_pixels + preproc_objects)
# preproc_study

In [45]:
# shortlist_df = pca_scored_df[
#     pca_scored_df["preprocessing"].isin(preproc_study)
# ].copy()

# shortlist_df = (
#     shortlist_df
#     .sort_values("selection_score", ascending=False)
#     .groupby("matrix_variant", group_keys=False)
#     .head(5)
#     .reset_index(drop=True)
# )

# shortlist_df[available_cols]

In [46]:
def join_unique(values) -> str:
    values = [str(v) for v in values if pd.notna(v)]
    return ", ".join(sorted(set(values)))


candidate_parts = []

# Best candidates within each matrix variant.
top_by_variant = (
    pca_scored_df
    .sort_values("selection_score", ascending=False)
    .groupby("matrix_variant", group_keys=False)
    .head(N_SHORTLIST_PER_MATRIX_VARIANT)
    .copy()
)
top_by_variant["selection_reason"] = "top_per_matrix_variant"
candidate_parts.append(top_by_variant)

# Best global candidates.
top_global = pca_scored_df.head(N_GLOBAL_SHORTLIST).copy()
top_global["selection_reason"] = "top_global"
candidate_parts.append(top_global)

candidate_pool_df = (
    pd.concat(candidate_parts, ignore_index=True, sort=False)
    .drop_duplicates(
        subset=[
            "matrix_family",
            "matrix_variant",
            "matrix_method",
            "balanced_pixel_strategy",
            "preprocessing",
        ]
    )
    .sort_values(["matrix_family", "matrix_variant", "selection_score"], ascending=[True, True, False])
    .reset_index(drop=True)
)

pca_selected_preprocessings_df = (
    candidate_pool_df
    .sort_values("selection_score", ascending=False)
    .groupby(["matrix_family", "preprocessing"], as_index=False)
    .agg(
        preprocessing_steps=("preprocessing_steps", "first"),
        best_selection_score=("selection_score", "max"),
        best_rank=("rank", "min"),
        best_matrix_variant=("matrix_variant", "first"),
        selected_from_variants=("matrix_variant", join_unique),
        selected_from_methods=("matrix_method", join_unique),
        selected_from_strategies=("balanced_pixel_strategy", join_unique),
        selection_reasons=("selection_reason", join_unique),
        best_selection_flag=("selection_flag", "first"),
    )
    .sort_values(["matrix_family", "best_rank", "best_selection_score"], ascending=[True, True, False])
    .reset_index(drop=True)
)

display(candidate_pool_df[available_ranking_cols].head(30))
display(pca_selected_preprocessings_df)

print("Selected preprocessing names:")
print(sorted(pca_selected_preprocessings_df["preprocessing"].unique()))

,rank,matrix_family,matrix_variant,matrix_method,balanced_pixel_strategy,preprocessing,selection_score,selection_flag,n_observations,n_label_almond,n_label_peanut,label_counts,evr_pc1,cum_pc3,ncomp_90,ncomp_95,fisher_pc1,fisher_pc2,mahalanobis_pc1_pc2_pc3,class_trace_ratio,batch_trace_ratio,class_over_batch_ratio,object_class_trace_ratio,object_batch_trace_ratio,object_over_intra_ratio,mean_intra_object_trace
0,5,object_matrix,object_mean,object_mean,not_applicable,absorbance_sg_d1,4.887345,weak_class_separation,394,214,180,"{'almond': 214, 'peanut': 180}",0.735538,0.961121,2,3,0.006383,0.697112,0.692340,0.055729,0.018513,3.010237,NaN,NaN,NaN,NaN
1,8,object_matrix,object_mean,object_mean,not_applicable,absorbance_sg_d2,4.314959,weak_class_separation,394,214,180,"{'almond': 214, 'peanut': 180}",0.846272,0.971727,2,3,0.000026,0.427958,0.023053,0.027787,0.022875,1.214738,NaN,NaN,NaN,NaN
2,10,object_matrix,object_mean,object_mean,not_applicable,absorbance_snv_sg_smooth,3.307124,batch_sensitive,394,214,180,"{'almond': 214, 'peanut': 180}",0.595743,0.917383,3,4,0.520939,1.553504,3.988180,0.343847,0.068605,5.011950,NaN,NaN,NaN,NaN
3,12,object_matrix,object_mean,object_mean,not_applicable,absorbance_msc,3.235770,batch_sensitive,394,214,180,"{'almond': 214, 'peanut': 180}",0.571238,0.893044,4,5,0.520037,1.513861,4.159365,0.346190,0.067272,5.146128,NaN,NaN,NaN,NaN
4,13,object_matrix,object_mean,object_mean,not_applicable,absorbance_snv,3.232337,batch_sensitive,394,214,180,"{'almond': 214, 'peanut': 180}",0.570045,0.892202,4,5,0.521457,1.518728,4.179036,0.347045,0.067360,5.152086,NaN,NaN,NaN,NaN
5,1,object_matrix,object_median,object_median,not_applicable,absorbance_sg_d1,18.187026,weak_class_separation,394,214,180,"{'almond': 214, 'peanut': 180}",0.686619,0.949383,2,4,0.033790,0.725992,0.820368,0.077787,0.016322,4.765700,NaN,NaN,NaN,NaN
6,2,object_matrix,object_median,object_median,not_applicable,absorbance_sg_d2,17.012127,weak_class_separation,394,214,180,"{'almond': 214, 'peanut': 180}",0.795240,0.941203,2,4,0.002126,0.512237,0.030267,0.033561,0.018759,1.789113,NaN,NaN,NaN,NaN
7,3,object_matrix,object_median,object_median,not_applicable,absorbance,6.473148,weak_class_separation,394,214,180,"{'almond': 214, 'peanut': 180}",0.943681,0.993841,1,2,0.079432,0.279653,0.990189,0.043054,0.041268,1.043270,NaN,NaN,NaN,NaN
8,4,object_matrix,object_median,object_median,not_applicable,absorbance_sg_smooth,6.472025,weak_class_separation,394,214,180,"{'almond': 214, 'peanut': 180}",0.945494,0.995330,1,2,0.079403,0.279254,0.994553,0.043018,0.041273,1.042268,NaN,NaN,NaN,NaN
9,18,object_matrix,object_median,object_median,not_applicable,snv_sg_smooth,2.801495,batch_sensitive,394,214,180,"{'almond': 214, 'peanut': 180}",0.452619,0.830746,5,6,0.708313,0.018647,2.902236,0.248807,0.055357,4.494580,NaN,NaN,NaN,NaN


,matrix_family,preprocessing,preprocessing_steps,best_selection_score,best_rank,best_matrix_variant,selected_from_variants,selected_from_methods,selected_from_strategies,selection_reasons,best_selection_flag
0,object_matrix,absorbance_sg_d1,absorbance+sg_d1,18.187026,1,object_median,"object_mean, object_median","object_mean, object_median",not_applicable,top_per_matrix_variant,weak_class_separation
1,object_matrix,absorbance_sg_d2,absorbance+sg_d2,17.012127,2,object_median,"object_mean, object_median","object_mean, object_median",not_applicable,top_per_matrix_variant,weak_class_separation
2,object_matrix,absorbance,absorbance,6.473148,3,object_median,object_median,object_median,not_applicable,top_per_matrix_variant,weak_class_separation
3,object_matrix,absorbance_sg_smooth,absorbance+sg_smooth,6.472025,4,object_median,object_median,object_median,not_applicable,top_per_matrix_variant,weak_class_separation
4,object_matrix,absorbance_snv_sg_smooth,absorbance+snv+sg_smooth,3.307124,10,object_mean,object_mean,object_mean,not_applicable,top_per_matrix_variant,batch_sensitive
5,object_matrix,absorbance_msc,absorbance+msc,3.235770,12,object_mean,object_mean,object_mean,not_applicable,top_per_matrix_variant,batch_sensitive
6,object_matrix,absorbance_snv,absorbance+snv,3.232337,13,object_mean,object_mean,object_mean,not_applicable,top_per_matrix_variant,batch_sensitive
7,object_matrix,snv_sg_smooth,snv+sg_smooth,2.801495,18,object_median,object_median,object_median,not_applicable,top_per_matrix_variant,batch_sensitive
8,pixel_matrix,absorbance_sg_d1,absorbance+sg_d1,4.653973,6,balanced_pixels_center,balanced_pixels_center,balanced_pixels,center,top_per_matrix_variant,weak_object_separation
9,pixel_matrix,absorbance_sg_d2,absorbance+sg_d2,4.366398,7,balanced_pixels_center,balanced_pixels_center,balanced_pixels,center,top_per_matrix_variant,weak_object_separation


Selected preprocessing names:
['absorbance', 'absorbance_msc', 'absorbance_sg_d1', 'absorbance_sg_d2', 'absorbance_sg_smooth', 'absorbance_snv', 'absorbance_snv_sg_d1', 'absorbance_snv_sg_smooth', 'raw', 'sg_smooth', 'snv', 'snv_sg_d1', 'snv_sg_smooth']


In [47]:
save_parquet(
    pca_selected_preprocessings_df,
    PCA_SELECTED_PREPROCESSINGS_PATH,
)

print("Saved selected PCA preprocessings:")
print(" -", PCA_SELECTED_PREPROCESSINGS_PATH)

Saved selected PCA preprocessings:
 - C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\03_pca_non_noisy_all\pca_selected_preprocessings.parquet


In [48]:
pca_protocol = {
    "db_h5_path": str(DB_H5_PATH),
    "results_dir": str(RESULTS_DIR),

    "wavelength_mode": WAVELENGTH_MODE,
    "use_wavelength_window": bool(USE_WAVELENGTH_WINDOW),
    "results_tag": RESULTS_TAG,
    "window_min_nm": WINDOW_MIN_NM if USE_WAVELENGTH_WINDOW else np.nan,
    "window_max_nm": WINDOW_MAX_NM if USE_WAVELENGTH_WINDOW else np.nan,
    "n_active_bands": int(len(wavelengths)) if wavelengths is not None else np.nan,

    "target_class": TARGET_CLASS,
    "reference_classes": list(REFERENCE_CLASSES),
    "pca_sample_kind": PCA_SAMPLE_KIND,
    "pca_allowed_batches": PCA_ALLOWED_BATCHES,

    "n_components": int(N_COMPONENTS),
    "m_balanced_pixels": int(M_BALANCED_PIXELS),
    "replace_balanced_pixels": bool(REPLACE_BALANCED_PIXELS),
    "random_state": int(RANDOM_STATE),
    "balanced_pixel_strategies": BALANCED_PIXEL_STRATEGIES,
    "run_all_pixels": bool(RUN_ALL_PIXELS),

    "sg_window_length": int(SG_WINDOW_LENGTH),
    "sg_polyorder": int(SG_POLYORDER),

    "n_shortlist_per_matrix_variant": int(N_SHORTLIST_PER_MATRIX_VARIANT),
    "n_global_shortlist": int(N_GLOBAL_SHORTLIST),

    "n_pca_combinations": int(len(pca_scored_df)),
    "n_selected_preprocessing_rows": int(len(pca_selected_preprocessings_df)),
    "n_selected_preprocessing_names": int(pca_selected_preprocessings_df["preprocessing"].nunique()),

    "pca_summary_path": str(PCA_SUMMARY_PATH),
    "pca_selected_preprocessings_path": str(PCA_SELECTED_PREPROCESSINGS_PATH),
}

pca_protocol_df = pd.DataFrame([pca_protocol])

print("PCA protocol summary:")
display(pca_protocol_df)

PCA protocol summary:


,db_h5_path,results_dir,wavelength_mode,use_wavelength_window,results_tag,window_min_nm,window_max_nm,n_active_bands,target_class,reference_classes,pca_sample_kind,pca_allowed_batches,n_components,m_balanced_pixels,replace_balanced_pixels,random_state,balanced_pixel_strategies,run_all_pixels,sg_window_length,sg_polyorder,n_shortlist_per_matrix_variant,n_global_shortlist,n_pca_combinations,n_selected_preprocessing_rows,n_selected_preprocessing_names,pca_summary_path,pca_selected_preprocessings_path
0,C:\Users\alixg\OneDrive - Université Paris-Dau...,C:\Users\alixg\OneDrive - Université Paris-Dau...,non_noisy_all,False,non_noisy_all,NaN,NaN,63,peanut,"[almond, peanut]",pure,"[1, 2, 3, 4]",20,40,False,42,"[random, center]",False,11,2,5,10,72,17,13,C:\Users\alixg\OneDrive - Université Paris-Dau...,C:\Users\alixg\OneDrive - Université Paris-Dau...


In [49]:
print("03_pca_exploration_selection.ipynb completed.")
print()
print("Essential outputs:")
print(" -", PCA_SUMMARY_PATH)
print(" -", PCA_SELECTED_PREPROCESSINGS_PATH)
print()
print("Summary:")
print(f" - Wavelength mode: {WAVELENGTH_MODE}")
print(f" - Active bands: {len(wavelengths) if wavelengths is not None else 'unknown'}")
print(f" - PCA object subset size: {len(object_db_pca)}")
print(f" - Valid preprocessing methods: {len(valid_preprocessing_configs)}")
print(f" - PCA combinations evaluated: {len(pca_scored_df)}")
print(f" - Selected preprocessing rows: {len(pca_selected_preprocessings_df)}")
print(f" - Selected preprocessing names: {pca_selected_preprocessings_df['preprocessing'].nunique()}")
print()
print("Best global candidate:")
display(pca_scored_df.head(1)[available_cols])
print()
print("Next notebook:")
print("04A_simca_preselection.ipynb")

03_pca_exploration_selection.ipynb completed.

Essential outputs:
 - C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\03_pca_non_noisy_all\pca_summary.parquet
 - C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\03_pca_non_noisy_all\pca_selected_preprocessings.parquet

Summary:
 - Wavelength mode: non_noisy_all
 - Active bands: 63
 - PCA object subset size: 394
 - Valid preprocessing methods: 18
 - PCA combinations evaluated: 72
 - Selected preprocessing rows: 17
 - Selected preprocessing names: 13

Best global candidate:


,matrix_variant,preprocessing,selection_score,selection_flag,n_observations,ncomp_90,ncomp_95,cum_pc3,class_trace_ratio,batch_trace_ratio,class_over_batch_ratio,fisher_pc1,fisher_pc2,mahalanobis_pc1_pc2_pc3,object_class_trace_ratio,object_batch_trace_ratio,object_over_intra_ratio,mean_intra_object_trace
0,object_median,absorbance_sg_d1,18.187026,weak_class_separation,394,2,4,0.949383,0.077787,0.016322,4.7657,0.03379,0.725992,0.820368,NaN,NaN,NaN,NaN



Next notebook:
04A_simca_preselection.ipynb
